# 05 · Closure Phase Time-Series

Reproduces key figures from Zheng & Fattahi (2026):

- **Fig. 4** — Closure phase time-series for uniform soil moisture with depth:
  desert (positive anomaly) and agricultural (negative anomaly) environments.
- **Fig. 5** — Effect of varying soil moisture with depth.
- **Fig. 6** — Effect of soil texture (sandy loam vs silty clay).

The closure phase is defined as the difference between the bandwidth-1
(nearest-neighbour) and full-network InSAR time-series.

---
**References**
- Zheng, Y., & Fattahi, H. (2026).
  *Modeling, prediction, and retrieval of surface soil moisture from InSAR closure phase.*
  Remote Sensing of Environment, 333, 115104.
  https://doi.org/10.1016/j.rse.2025.115104
- De Zan, F., Parizzi, A., Prats-Iraola, P., & López-Dekker, P. (2014).
  *A SAR interferometric model for soil moisture.*
  IEEE Transactions on Geoscience and Remote Sensing, 52(1), 418–425.
  https://doi.org/10.1109/TGRS.2013.2241069

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from navasar.closure import closure_phase_timeseries

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

# Base simulation parameters (Zheng & Fattahi 2026, Table 1)
SIM_KW = dict(n_pixels=400, n_layers=200, max_depth=0.30,
              sand=0.51, clay=0.13, rng=np.random.default_rng(42))

## Synthetic moisture time-series: desert vs agricultural

In [ ]:
T = 30   # number of acquisitions
t = np.arange(T)

# Desert: low baseline, brief positive rain anomaly (t=8..12)
mv_desert = np.full(T, 0.05)
mv_desert[8:13] = 0.18

# Agricultural: high baseline, slow drying negative anomaly (t=15..25)
mv_agri = np.full(T, 0.30)
mv_agri[15:26] = np.linspace(0.30, 0.10, 11)

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(t, mv_desert, 'b-o', ms=4)
axes[0].set_ylabel('$m_v$'); axes[0].set_title('Desert (Mojave-like)')
axes[0].grid(True, alpha=0.3)
axes[1].plot(t, mv_agri, 'g-o', ms=4)
axes[1].set_ylabel('$m_v$'); axes[1].set_title('Agricultural (Central Valley-like)')
axes[1].set_xlabel('Acquisition index')
axes[1].grid(True, alpha=0.3)
plt.suptitle('Input soil moisture time-series', y=1.01)
plt.tight_layout()
plt.show()

## Fig. 4 — Closure phase time-series: uniform moisture with depth (Zheng & Fattahi 2026)

In [ ]:
for freq, band in [(1.4, 'L'), (5.0, 'C')]:
    cp_desert, bw1_d, fn_d = closure_phase_timeseries(
        mv_desert, freq_ghz=freq, **SIM_KW)
    cp_agri, bw1_a, fn_a = closure_phase_timeseries(
        mv_agri, freq_ghz=freq, **SIM_KW)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

    axes[0].plot(t, cp_desert, 'b-o', ms=4)
    axes[0].axhline(0, color='k', lw=0.8, ls='--')
    axes[0].set_xlabel('Acquisition index')
    axes[0].set_ylabel('Closure phase [deg]')
    axes[0].set_title(f'Desert — {band}-band\n(positive anomaly → positive step)')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t, cp_agri, 'g-o', ms=4)
    axes[1].axhline(0, color='k', lw=0.8, ls='--')
    axes[1].set_xlabel('Acquisition index')
    axes[1].set_ylabel('Closure phase [deg]')
    axes[1].set_title(f'Agricultural — {band}-band\n(negative anomaly → negative step)')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'Closure phase time-series — uniform moisture with depth\n'
                 f'(Zheng & Fattahi 2026, Fig. 4, {band}-band)', y=1.02)
    plt.tight_layout()
    plt.savefig(f'../examples/fig04_closure_uniform_{band}band.png', dpi=150)
    plt.show()

## Fig. 5 — Varying soil moisture with depth (Zheng & Fattahi 2026)

In [ ]:
# Desert: moisture increases linearly with depth
# shape (T, N): each row is the depth profile at acquisition t
N = 200
mv_desert_depth = np.zeros((T, N))
for ti in range(T):
    base = mv_desert[ti]
    mv_desert_depth[ti] = np.linspace(base, base * 1.5, N)

# Agricultural: moisture decreases linearly with depth
mv_agri_depth = np.zeros((T, N))
for ti in range(T):
    base = mv_agri[ti]
    mv_agri_depth[ti] = np.linspace(base, base * 0.6, N)

cp_d_unif, _, _ = closure_phase_timeseries(mv_desert, freq_ghz=1.4, **SIM_KW)
cp_d_vary, _, _ = closure_phase_timeseries(mv_desert_depth, freq_ghz=1.4, **SIM_KW)
cp_a_unif, _, _ = closure_phase_timeseries(mv_agri, freq_ghz=1.4, **SIM_KW)
cp_a_vary, _, _ = closure_phase_timeseries(mv_agri_depth, freq_ghz=1.4, **SIM_KW)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t, cp_d_unif, 'b--', label='Uniform depth')
axes[0].plot(t, cp_d_vary, 'b-',  label='Varying depth')
axes[0].axhline(0, color='k', lw=0.8, ls=':')
axes[0].set_title('Desert — L-band'); axes[0].set_xlabel('Acquisition')
axes[0].set_ylabel('Closure phase [deg]'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t, cp_a_unif, 'g--', label='Uniform depth')
axes[1].plot(t, cp_a_vary, 'g-',  label='Varying depth')
axes[1].axhline(0, color='k', lw=0.8, ls=':')
axes[1].set_title('Agricultural — L-band'); axes[1].set_xlabel('Acquisition')
axes[1].set_ylabel('Closure phase [deg]'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Effect of varying soil moisture with depth\n(Zheng & Fattahi 2026, Fig. 5)', y=1.02)
plt.tight_layout()
plt.savefig('../examples/fig05_closure_varying_depth.png', dpi=150)
plt.show()

## Fig. 6 — Soil texture effect (Zheng & Fattahi 2026)

In [ ]:
textures = {
    'Sandy loam (51% sand, 13% clay)': dict(sand=0.51, clay=0.13),
    'Silty clay (5% sand, 47% clay)':  dict(sand=0.05, clay=0.47),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ['b', 'r']

for ax, (mv_ts, env) in zip(axes, [(mv_desert, 'Desert'), (mv_agri, 'Agricultural')]):
    for (label, tex), col in zip(textures.items(), colors):
        kw = dict(n_pixels=400, n_layers=200, max_depth=0.30,
                  rng=np.random.default_rng(42), freq_ghz=1.4, **tex)
        cp, _, _ = closure_phase_timeseries(mv_ts, **kw)
        ax.plot(t, cp, color=col, label=label)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_title(f'{env} — L-band')
    ax.set_xlabel('Acquisition index')
    ax.set_ylabel('Closure phase [deg]')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of soil texture on closure phase\n(Zheng & Fattahi 2026, Fig. 6)', y=1.02)
plt.tight_layout()
plt.savefig('../examples/fig06_closure_soil_texture.png', dpi=150)
plt.show()